In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/processed/2_preprocess/step2_death.csv", low_memory=False)
df["BIRTH_YMD"]  = pd.to_datetime(df["BIRTH_YMD"], errors="coerce")
df["ABATT_DATE"] = pd.to_datetime(df["ABATT_DATE"], errors="coerce")
df["JUDGE_DATE"] = pd.to_datetime(df["JUDGE_DATE"], errors="coerce")

print(f"로드 완료: {df.shape}")

# area 컬럼 한번에 확인
for col in ["C2023", "C2024", "C2025", "AREA"]:
    print(f"\n=== {col} ===")
    print(f"결측 수: {df[col].isnull().sum():,}")
    print(df[col].describe())

로드 완료: (2408699, 44)

=== C2023 ===
결측 수: 37,384
count    2.371315e+06
mean     1.493923e+02
std      2.192661e+02
min      0.000000e+00
25%      4.600000e+01
50%      9.300000e+01
75%      1.740000e+02
max      2.829000e+03
Name: C2023, dtype: float64

=== C2024 ===
결측 수: 37,384
count    2.371315e+06
mean     1.425661e+02
std      2.101982e+02
min      0.000000e+00
25%      4.100000e+01
50%      8.900000e+01
75%      1.680000e+02
max      2.765000e+03
Name: C2024, dtype: float64

=== C2025 ===
결측 수: 37,384
count    2.371315e+06
mean     1.344164e+02
std      2.082581e+02
min      0.000000e+00
25%      3.500000e+01
50%      8.100000e+01
75%      1.590000e+02
max      2.603000e+03
Name: C2025, dtype: float64

=== AREA ===
결측 수: 37,384
count    2.371315e+06
mean     1.332739e+03
std      2.459444e+03
min      0.000000e+00
25%      0.000000e+00
50%      7.000000e+02
75%      1.708000e+03
max      4.017600e+04
Name: AREA, dtype: float64


In [2]:
# 0값 확인
for col in ["C2023", "C2024", "C2025", "AREA"]:
    zero_cnt = (df[col] == 0).sum()
    print(f"{col} 0인 행 수: {zero_cnt:,}")

# AREA 0인 농장 확인
print("\nAREA 0인 행 샘플:")
print(df[df['AREA'] == 0][['AREA', 'C2023', 'C2024', 'C2025', 'FARM_UNIQUE_NO']].head(10))

C2023 0인 행 수: 33,209
C2024 0인 행 수: 58,840
C2025 0인 행 수: 114,357
AREA 0인 행 수: 811,010

AREA 0인 행 샘플:
    AREA  C2023  C2024  C2025            FARM_UNIQUE_NO
40   0.0  138.0  152.0  160.0  69vGOAYpMyW0t++JSnF8yA==
41   0.0   74.0   77.0   84.0  DLLjxlPOxoFC0q7LZ4fftg==
42   0.0  603.0  574.0  579.0  BQLgYbfFs8BRGRFlHzjl2w==
43   0.0  395.0    0.0    0.0  D4PHb6d8Dz2Ya7xsNVRgQA==
44   0.0  245.0  232.0  223.0  QH5tdsbw6T9qBD9w7t9fgA==
45   0.0  163.0  169.0  185.0  DFRYZRonPNgMqG0bhY/qQQ==
59   0.0    5.0    7.0    0.0  8IpXZB29jRCW/NhoiCWz7Q==
62   0.0   20.0    7.0    3.0  ZUYquOma++RCJVZSVemVFg==
67   0.0  148.0  143.0  125.0  zN7GtowZ0xnqiRhVrimmNA==
75   0.0   29.0   28.0   34.0  r38OEsexaDRTYCsMxwNRKA==


In [3]:
raw_area = pd.read_csv("../../data/raw/hanwoo_area.csv")
print("원본 AREA 0 개수:", (raw_area['AREA'] == 0).sum())
print("원본 AREA -99 개수:", (raw_area['AREA'] == -99).sum())
print("원본 AREA 분포:")
print(raw_area['AREA'].describe())

원본 AREA 0 개수: 4
원본 AREA -99 개수: 29043
원본 AREA 분포:
count    91896.000000
mean       492.294381
std        876.690968
min        -99.000000
25%        -99.000000
50%        243.000000
75%        704.732500
max      48360.000000
Name: AREA, dtype: float64


In [4]:
# 원본에서 AREA가 0인 농장이 병합 후에도 0인지 확인
zero_farms_raw = raw_area[raw_area['AREA'] == 0]['FARM_UNIQUE_NO'].tolist()
print(f"원본 AREA 0인 농장: {zero_farms_raw}")
print(df[df['FARM_UNIQUE_NO'].isin(zero_farms_raw)][['AREA','C2023','C2024','C2025']].head())

# 병합 후 AREA 0인데 원본에서 -99였던 케이스 확인
print(f"\n병합 후 AREA 0인 행의 원본 확인:")
sample_farm = df[df['AREA'] == 0]['FARM_UNIQUE_NO'].iloc[0]
print(raw_area[raw_area['FARM_UNIQUE_NO'] == sample_farm][['FARM_UNIQUE_NO','C2023','C2024','C2025','AREA']])

원본 AREA 0인 농장: ['pA1sPRByzwAJsEwvl45qBw==', 'NEhPWejQju1SS9gl+PunFw==', 'nhbMCbufVbCSzH4GZiuwTg==', 'LAT1rq/7BUsPHcOeLl17iQ==']
        AREA  C2023  C2024  C2025
57040    0.0    4.0    3.0    2.0
254955   0.0    4.0    3.0    2.0
333268   0.0   10.0    7.0    0.0
491535   0.0    8.0    6.0    5.0
492174   0.0    8.0    6.0    5.0

병합 후 AREA 0인 행의 원본 확인:
                 FARM_UNIQUE_NO  C2023  C2024  C2025  AREA
13096  69vGOAYpMyW0t++JSnF8yA==  138.0  152.0  160.0 -99.0


In [5]:
# AREA 0 → NaN
df['AREA'] = df['AREA'].replace(0, np.nan)

# C2023~C2025 0도 같은 이유인지 확인 후 처리
for col in ["C2023", "C2024", "C2025"]:
    sample_farm = df[df[col] == 0]['FARM_UNIQUE_NO'].iloc[0]
    raw_val = raw_area[raw_area['FARM_UNIQUE_NO'] == sample_farm][col].values
    print(f"{col} 0인 샘플 농장 원본값: {raw_val}")

C2023 0인 샘플 농장 원본값: [-99.]
C2024 0인 샘플 농장 원본값: [-99.]
C2025 0인 샘플 농장 원본값: [-99.]


In [6]:
# C2023~C2025, AREA 0 → NaN
for col in ["C2023", "C2024", "C2025", "AREA"]:
    zero_cnt = (df[col] == 0).sum()
    df[col] = df[col].replace(0, np.nan)
    print(f"{col}: 0 → NaN {zero_cnt:,}개, 처리 후 결측: {df[col].isnull().sum():,}")

C2023: 0 → NaN 33,209개, 처리 후 결측: 70,593
C2024: 0 → NaN 58,840개, 처리 후 결측: 96,224
C2025: 0 → NaN 114,357개, 처리 후 결측: 151,741
AREA: 0 → NaN 0개, 처리 후 결측: 848,394


In [7]:
print(f"AREA 0인 행 수: {(df['AREA'] == 0).sum():,}")
print(f"AREA 결측 수: {df['AREA'].isnull().sum():,}")
print(df['AREA'].value_counts().head(10))

AREA 0인 행 수: 0
AREA 결측 수: 848,394
AREA
800.0     10350
1200.0     7697
1500.0     6827
1600.0     6500
350.0      6063
1000.0     6052
700.0      5845
1300.0     5827
1400.0     5660
400.0      5532
Name: count, dtype: int64


In [8]:
# C2023 결측 원인 분류
raw_area = pd.read_csv("../../data/raw/hanwoo_area.csv")
area_farms = set(raw_area['FARM_UNIQUE_NO'])

null_c2023 = set(df[df['C2023'].isnull()]['FARM_UNIQUE_NO'])
not_in_area = len(null_c2023 - area_farms)
in_area_but_null = len(null_c2023 & area_farms)

print(f"C2023 결측 원인:")
print(f"  area_df에 없는 농장: {not_in_area:,}")
print(f"  area_df에 있는데 결측 (원본 -99): {in_area_but_null:,}")
print(f"  합계: {not_in_area + in_area_but_null:,}")

C2023 결측 원인:
  area_df에 없는 농장: 3,077
  area_df에 있는데 결측 (원본 -99): 1,484
  합계: 4,561


In [9]:
# 행 단위로 재확인
null_rows = df[df['C2023'].isnull()]

not_in_area_rows = null_rows[~null_rows['FARM_UNIQUE_NO'].isin(area_farms)]
in_area_but_null_rows = null_rows[null_rows['FARM_UNIQUE_NO'].isin(area_farms)]

print(f"C2023 결측 행 원인:")
print(f"  area_df에 없는 농장 소속 행: {len(not_in_area_rows):,}")
print(f"  area_df에 있는데 결측인 행: {len(in_area_but_null_rows):,}")
print(f"  합계: {len(not_in_area_rows) + len(in_area_but_null_rows):,}")
print(f"  실제 결측: {df['C2023'].isnull().sum():,}")

C2023 결측 행 원인:
  area_df에 없는 농장 소속 행: 37,384
  area_df에 있는데 결측인 행: 33,209
  합계: 70,593
  실제 결측: 70,593


In [10]:
df.to_csv("../../data/processed/2_preprocess/step3_area.csv",
          index=False, encoding="utf-8-sig")
print(f"저장 완료: {df.shape}")

저장 완료: (2408699, 44)
